In [1]:
import pandas as pd 
import Bio as SeqIO

In [12]:
gff_file = "../data/Oryza_sativa.IRGSP-1.0.63.gff3"
pep_file = "../data/Oryza_sativa.IRGSP-1.0.pep.all.fa"

In [13]:
with open(gff_file) as f:
    for i in range(10):
        print(f.readline())

##gff-version 3

##sequence-region   1 1 43270923

##sequence-region   10 1 23207287

##sequence-region   11 1 29021106

##sequence-region   12 1 27531856

##sequence-region   2 1 35937250

##sequence-region   3 1 36413819

##sequence-region   4 1 35502694

##sequence-region   5 1 29958434

##sequence-region   6 1 31248787



In [ ]:
#These ## lines are header metadata, not gene information.

In [15]:
#find gene entry 
with open(gff_file) as f:
    for line in f:
        if not line.startswith('#'):
            print(line)
            break
        
    

1	IRGSP-1.0	chromosome	1	43270923	.	.	.	ID=chromosome:1;Alias=Chr1,AP014957.1,NC_029256.1



In [ ]:
#This is not yet a gene. It's the chromosome definition.the above result.


In [34]:
with open(gff_file) as f:
    for line in f:

        # Skip header lines
        if line.startswith("#"):
            continue

        # Split the current line into columns
        columns = line.strip().split("\t")

        # Check if this row represents a gene
        if columns[2] == "gene":
            print(columns)
            break




        

['1', 'RAP2022-09-01', 'gene', '2983', '10815', '.', '+', '.', 'ID=gene:Os01g0100100;biotype=protein_coding;gene_id=Os01g0100100;logic_name=rapdb_genes']


In [ ]:
Why are we doing this?

A GFF3-General Feature Format Version 3, file contains many different feature types:

gene
mRNA
exon
CDS
five_prime_UTR
three_prime_UTR

We only want gene entries for our first database.

In [ ]:
After you get the first gene line

We'll write a parser that extracts:

Column	Example
Chromosome	1
Feature	gene
Start	2983
End	10815
Strand	+
Gene ID	Os01g0100100

and store everything in a pandas DataFrame.

In [40]:
#extract the genes now
genes=[]

with open(gff_file) as f:
    for line in f:
        if line.startswith('#'):
            continue
        columns= line.strip().split("\t")
        if columns[2]=="gene":

         gene={ "Chromosome": columns[0],
                "Start": int(columns[3]),
                "End": int(columns[4]),
                "Strand": columns[6],
                "Attributes": columns[8]}
         genes.append(gene)
              

In [41]:
print(len(genes))

35806


In [45]:
genes[2]

{'Chromosome': '1',
 'Start': 11372,
 'End': 12284,
 'Strand': '-',
 'Attributes': 'ID=gene:Os01g0100300;biotype=protein_coding;gene_id=Os01g0100300;logic_name=rapdb_genes'}

In [46]:
#convert this into dataframe to store it in an excel like format
import pandas as pd
df= pd.DataFrame(genes)

df.head

<bound method NDFrame.head of       Chromosome   Start     End Strand  \
0              1    2983   10815      +   
1              1   11218   12435      +   
2              1   11372   12284      -   
3              1   12721   15685      +   
4              1   12808   13978      -   
...          ...     ...     ...    ...   
35801         Pt  126704  127174      +   
35802         Pt  127479  128255      +   
35803         Pt  132154  132435      +   
35804         Pt  132454  132844      +   
35805         Pt  134200  134481      +   

                                              Attributes  
0      ID=gene:Os01g0100100;biotype=protein_coding;ge...  
1      ID=gene:Os01g0100200;biotype=protein_coding;ge...  
2      ID=gene:Os01g0100300;biotype=protein_coding;ge...  
3      ID=gene:Os01g0100400;biotype=protein_coding;ge...  
4      ID=gene:Os01g0100466;biotype=protein_coding;ge...  
...                                                  ...  
35801  ID=gene:gene-rps7-2;Name=rps7;bio

In [ ]:
#the attribute column looks messy 
#use string parsing
#the attribute column is actually a series of key =value pairs separated by semicolon
# we will transform it into python dictionary 

In [ ]:
Entire string

↓

Split by ;

↓

Individual fields

↓

Split each field by =

↓

Key + Value
#hierarchical parsing 

In [70]:
genes=[]

with open(gff_file) as f:
    for line in f:
        if line.startswith("#"):
         continue
        columns=line.strip().split("\t")
        if columns[2]=="gene":
            #parse the attribute column
            attribute_dict={}
            for items in columns[8].split(";"):
                key,value=items.split("=")
                attribute_dict[key]=value


            #building a clean gene record

            gene={"Gene_ID": attribute_dict["gene_id"],
                "Chromosome": columns[0],
                "Start": int(columns[3]),
                "End": int(columns[4]),
                "Strand": columns[6],
                "Biotype": attribute_dict["biotype"]
            }
            genes.append(gene)
print(len(genes))       

35806


In [71]:
genes[0]

{'Gene_ID': 'Os01g0100100',
 'Chromosome': '1',
 'Start': 2983,
 'End': 10815,
 'Strand': '+',
 'Biotype': 'protein_coding'}

In [74]:
#add gene length as well
gene = {
    "Gene_ID": attribute_dict["gene_id"],
    "Chromosome": columns[0],
    "Start": int(columns[3]),
    "End": int(columns[4]),
    "Length": int(columns[4]) - int(columns[3]) + 1,
    "Strand": columns[6],
    "Biotype": attribute_dict["biotype"]
}

In [76]:
genes[0]

{'Gene_ID': 'Os01g0100100',
 'Chromosome': '1',
 'Start': 2983,
 'End': 10815,
 'Strand': '+',
 'Biotype': 'protein_coding'}

In [77]:
import pandas as pd

df = pd.DataFrame(genes)

In [78]:
df.head()

,Gene_ID,Chromosome,Start,End,Strand,Biotype
0,Os01g0100100,1,2983,10815,+,protein_coding
1,Os01g0100200,1,11218,12435,+,protein_coding
2,Os01g0100300,1,11372,12284,-,protein_coding
3,Os01g0100400,1,12721,15685,+,protein_coding
4,Os01g0100466,1,12808,13978,-,protein_coding


In [79]:
len(df)

35806

In [80]:
df.columns

Index(['Gene_ID', 'Chromosome', 'Start', 'End', 'Strand', 'Biotype'], dtype='object')

In [81]:
df.to_csv(
    "../data/rice_gene_database.csv",
    index=False
)

In [83]:
pep_file="../data/Oryza_sativa.IRGSP-1.0.pep.all.fa"

In [84]:
with open(pep_file) as f:
    for line in f:
        print(f.readline().strip())

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



DFSVYKPSKKPPAADQSETLAAAAVVP
MGLCLSSGVAAAMAEAGMAASTAMVLLPTGELREYPRPATAARVLEDVAAAEGEEEDVGR
AASSSSTTSGCGGGRRRRGSVAPLVFAPPEEEYEYDASDYCKSNASAAAAAAGKRRPVAA
>Os01t0350300-00 pep chromosome:IRGSP-1.0:1:13980666:13983744:-1 gene:Os01g0350300 transcript:Os01t0350300-00 gene_biotype:protein_coding transcript_biotype:protein_coding
RFDEQAKAWSGHARELACDAALLARRADGSPDAASAATVKALLERAADLSRRRPRPTAAV
TLAKMVYDTLRPRFDCGAFVSVSAINPDMAMVFMRMLRQLDDDDKHESVGGEEPSVSGEA
YELKPLSAENSKILFLQRIFGHDNKICLDDEFAEVADKILKKCDGVPIAILALASLLAGK
>Os09t0500200-01 pep chromosome:IRGSP-1.0:9:19357923:19360992:-1 gene:Os09g0500200 transcript:Os09t0500200-01 gene_biotype:protein_coding transcript_biotype:protein_coding
WICWLNGQRKKAPTPEELAELEARRERVKQNIELLKKKEEEEKKTGVRPVKTVGKFESPN
>Os05t0584400-01 pep chromosome:IRGSP-1.0:5:29077046:29078529:-1 gene:Os05g0584400 transcript:Os05t0584400-01 gene_biotype:protein_coding transcript_biotype:protein_coding
SDDSDDDCTDLDFIISLHRSRSASPSYDTLFFAAAASEPSTKASFQPSHHFCAKRRGGGG
LIRVAISLLLILMVTKTNERPITSLNEIV

In [ ]:
##we want this {
"Protein1":"AAAABBBBCCCC",
"Protein2":"DDDDEEEE"
##}
Current protein ID
Protein1

and

Current sequence
AAAABBBBCCCC

So we need two variables.

current_header = None

current_sequence = ""

#also, before moving to the next protein we must save the previous one.
#and we save it in a dictionary c

In [ ]:
Read line

↓

Does it start with ">" ?

        YES
         │
         ▼
Save previous protein
Start new protein

        NO
         │
         ▼
Append sequence

In [ ]:
#algorithm
If the line starts with >:
If we were already reading a protein, save it.
Set the new current_header.
Reset current_sequence to an empty string.
Otherwise:
Add the sequence line to current_sequence.

In [86]:
protein={}
current_header= None
Current_seq= "" 

In [89]:
with open(pep_file) as f:
    for line in f:
        line= line.strip()
        if line.startswith(">"):
            #save the previous protein
            if current_header is not None:
                protein[current_header]= current_seq
            #extract gene id
            fields= line.split()
            for field in fields:
                 if field.startswith("gene"):
                     current_header= field.replace("gene:", "")
                     break
            current_seq= ""
        else:
            current_seq+= line
protein[current_header]= current_seq
        

In [91]:
print(len(protein))

35804


In [92]:
list(protein.keys())[:5]

['gene-rps2', 'gene-petB', 'gene-ndhB-2', 'gene-rpl36', 'gene-rpoC1']

In [94]:
# Add protein sequences to gene database
df["Protein_Sequence"] = df["Gene_ID"].map(protein)

# Calculate protein length
df["Protein_Length"] = df["Protein_Sequence"].str.len()

# Check for genes without protein sequences
missing = df["Protein_Sequence"].isna().sum()
print(f"Genes without protein sequence: {missing}")

# Save the integrated database
df.to_csv("../data/rice_gene_database.csv", index=False)

# Preview
df.head()

Genes without protein sequence: 2


,Gene_ID,Chromosome,Start,End,Strand,Biotype,Protein_Sequence,Protein_Length
0,Os01g0100100,1,2983,10815,+,protein_coding,MSSAAGQDNGDTAGDYIKWMCGAGGRAGGAMANLQRGVGSLVRDIG...,702.0
1,Os01g0100200,1,11218,12435,+,protein_coding,MEEAGERDADETHAWSGTASPAALWKTVASSAAMLKLALAMISAAF...,142.0
2,Os01g0100300,1,11372,12284,-,protein_coding,VCQLAHYIVTAQPHAHVRERPRHGRGRRVGSPPLHRRPCLLRHQPQ...,269.0
3,Os01g0100400,1,12721,15685,+,protein_coding,MDSIRRRSAGGILGILFLVLLRWAGAGDPYAYYEWEVSYVWGAPLG...,593.0
4,Os01g0100466,1,12808,13978,-,protein_coding,MPQFVPPTPSCQGLLRCCTPCHVSSSGSSRPFLTFTTRFQLVVTFS...,77.0


In [99]:
df["Protein_Length"] = df["Protein_Sequence"].str.len()

In [100]:
#check for missing protein
missing = df["Protein_Sequence"].isna().sum()

In [101]:
df.to_csv("../data/rice_gene_database.csv", index=False)

In [ ]:
PlantGPT
│
├── Phase 1: Data Preparation ✅
│   ├── Parse GFF3
│   ├── Parse Protein FASTA
│   ├── Merge gene + protein data
│   └── Build rice_gene_database.csv
│
├── Phase 2: Functional Annotation  ← Next
│   ├── GO Terms
│   ├── Gene descriptions
│   ├── Pathways
│   └── Gene symbols
│
├── Phase 3: Semantic Search
│   ├── Sentence embeddings
│   ├── FAISS vector database
│   └── Similarity search
│
├── Phase 4: PlantGPT (RAG)
│   ├── LangChain
│   ├── LLM
│   └── Question answering
│
└── Phase 5: AI Gene Prioritization
    ├── Expression data
    ├── Network features
    ├── ESM2 embeddings
    └── ML ranking

In [110]:
annotations= pd.read_csv("../data/annotation.txt")

In [111]:
print(annotations.head())
print(annotations.columns)
print(annotations.shape)

    Gene stable ID Transcript stable ID Gene description GO term accession  \
0  ENSRNA049474694   ENSRNA049474694-T1              NaN        GO:0003735   
1  ENSRNA049474694   ENSRNA049474694-T1              NaN        GO:0005840   
2  ENSRNA049474700   ENSRNA049474700-T1              NaN        GO:0003735   
3  ENSRNA049474700   ENSRNA049474700-T1              NaN        GO:0005840   
4  ENSRNA049474705   ENSRNA049474705-T1              NaN        GO:0003735   

                         GO term name  
0  structural constituent of ribosome  
1                            ribosome  
2  structural constituent of ribosome  
3                            ribosome  
4  structural constituent of ribosome  
Index(['Gene stable ID', 'Transcript stable ID', 'Gene description',
       'GO term accession', 'GO term name'],
      dtype='object')
(174425, 5)


In [112]:
annotations[
    annotations.iloc[:,0].astype(str).str.startswith("Os")
].head()

,Gene stable ID,Transcript stable ID,Gene description,GO term accession,GO term name
1998,Os02g0788600,Os02t0788600-01,NaN,GO:0005634,nucleus
1999,Os02g0788600,Os02t0788600-01,NaN,GO:0006457,protein folding
2000,Os02g0788600,Os02t0788600-01,NaN,GO:0044183,protein folding chaperone
2001,Os06g0238000,Os06t0238000-00,NaN,GO:0005515,protein binding
2002,Os06g0238000,Os06t0238000-00,NaN,GO:0004842,ubiquitin-protein transferase activity


In [113]:
print(df["Gene_ID"].head()) #pick one gene it and see if it matches w any in the annotaion file

0    Os01g0100100
1    Os01g0100200
2    Os01g0100300
3    Os01g0100400
4    Os01g0100466
Name: Gene_ID, dtype: object


In [114]:
annotations[
    annotations.iloc[:,0] == "Os01g0100100"
]

,Gene stable ID,Transcript stable ID,Gene description,GO term accession,GO term name
22692,Os01g0100100,Os01t0100100-01,NaN,GO:0005096,GTPase activator activity


In [115]:
annotations.columns = [
    "Gene_ID",
    "Transcript_ID",
    "Description",
    "GO_ID",
    "GO_Term"
]

In [117]:
annotations.head

<bound method NDFrame.head of                 Gene_ID       Transcript_ID  \
0       ENSRNA049474694  ENSRNA049474694-T1   
1       ENSRNA049474694  ENSRNA049474694-T1   
2       ENSRNA049474700  ENSRNA049474700-T1   
3       ENSRNA049474700  ENSRNA049474700-T1   
4       ENSRNA049474705  ENSRNA049474705-T1   
...                 ...                 ...   
174420       gene-mat-r    transcript-mat-r   
174421       gene-mat-r    transcript-mat-r   
174422       gene-mat-r    transcript-mat-r   
174423       gene-mat-r    transcript-mat-r   
174424      gene-rps3-2   transcript-rps3-2   

                                              Description       GO_ID  \
0                                                     NaN  GO:0003735   
1                                                     NaN  GO:0005840   
2                                                     NaN  GO:0003735   
3                                                     NaN  GO:0005840   
4                                       

In [118]:
annotations[
    annotations["Gene_ID"]=="Os02g0788600"
]

,Gene_ID,Transcript_ID,Description,GO_ID,GO_Term
1998,Os02g0788600,Os02t0788600-01,NaN,GO:0005634,nucleus
1999,Os02g0788600,Os02t0788600-01,NaN,GO:0006457,protein folding
2000,Os02g0788600,Os02t0788600-01,NaN,GO:0044183,protein folding chaperone


In [ ]:
#same gene appear multiple timeas as they could have diff GO terms. But we want one gene each in our data set only 

In [119]:
print("Total rows:", len(annotations))
print("Unique genes:", annotations["Gene_ID"].nunique())

Total rows: 174425
Unique genes: 38993


In [122]:
#combine all Go terms belonging to one gene by aggregating them 
annotations_grouped=( annotations.groupby("Gene_ID").agg({
                      "Transcript_ID": "first",
        "Description": "first",
        "GO_ID": lambda x: "; ".join(x.dropna().unique()),
        "GO_Term": lambda x: "; ".join(x.dropna().unique())
    })
    .reset_index()
                    )                          

In [123]:
annotations_grouped.head()

,Gene_ID,Transcript_ID,Description,GO_ID,GO_Term
0,ENSRNA049440515,ENSRNA049440515-T1,None,GO:0030533; GO:0006412,triplet codon-amino acid adaptor activity; tra...
1,ENSRNA049440716,ENSRNA049440716-T1,None,GO:0030533; GO:0006412,triplet codon-amino acid adaptor activity; tra...
2,ENSRNA049441102,ENSRNA049441102-T1,None,GO:0030533; GO:0006412,triplet codon-amino acid adaptor activity; tra...
3,ENSRNA049441259,ENSRNA049441259-T1,None,GO:0030533; GO:0006412,triplet codon-amino acid adaptor activity; tra...
4,ENSRNA049441339,ENSRNA049441339-T1,None,GO:0030533; GO:0006412,triplet codon-amino acid adaptor activity; tra...


In [125]:
print("Total genes:", len(annotations_grouped))

print("Os genes:",
      annotations_grouped["Gene_ID"].str.startswith("Os").sum())

print("ENS genes:",
      annotations_grouped["Gene_ID"].str.startswith("ENS").sum())

Total genes: 38993
Os genes: 37864
ENS genes: 949


In [126]:
#keep only the os gene ids
annotations_grouped = annotations_grouped[
    annotations_grouped["Gene_ID"].str.startswith("Os")
].copy()

In [127]:
print(annotations_grouped.shape)
annotations_grouped.head()

(37864, 5)


,Gene_ID,Transcript_ID,Description,GO_ID,GO_Term
1017,Os01g0100100,Os01t0100100-01,None,GO:0005096,GTPase activator activity
1018,Os01g0100200,Os01t0100200-01,None,,
1019,Os01g0100300,Os01t0100300-00,None,GO:0005506; GO:0020037; GO:0016705; GO:0004497...,iron ion binding; heme binding; oxidoreductase...
1020,Os01g0100400,Os01t0100400-01,None,GO:0016491; GO:0005507,oxidoreductase activity; copper ion binding
1021,Os01g0100466,Os01t0100466-00,None,,


In [128]:
#merge 
df = df.merge(
    annotations_grouped,
    on="Gene_ID",
    how="left"
)

In [129]:
print(df.shape)

(35806, 12)


In [130]:
print("Genes with GO terms:",
      df["GO_Term"].notna().sum())

print("Genes without GO terms:",
      df["GO_Term"].isna().sum())

Genes with GO terms: 35695
Genes without GO terms: 111


In [ ]:
one merged master database
Gene_ID
      │
      ├── Chromosome
      ├── Start
      ├── End
      ├── Strand
      ├── Protein sequence
      ├── Protein length
      ├── GO IDs
      ├── GO Terms
      └── Description

In [132]:
#save it
df.to_csv("../data/rice_gene_database.csv", index=False)

print("Database saved successfully!")

Database saved successfully!
